In [1]:
text = """
The Camera is Excellent.
The Camera is bad.
"""

In [2]:
text

'\nThe Camera is Excellent.\nThe Camera is bad.\n'

In [3]:
tokens = text.split()

In [4]:
tokens

['The', 'Camera', 'is', 'Excellent.', 'The', 'Camera', 'is', 'bad.']

In [5]:
vocab = {
    "<PAD>": 0,
    "<UNK>": 1,
    "the": 2,
    "camera": 3,
    "is": 4,
    "good": 5,
    "excellent": 6,
    "battery": 7,
    "bad": 8,
    "terrible": 9
}

In [6]:
tokens = text.split()
token_ids = [
    vocab.get(token.lower(), vocab["<UNK>"])
    for token in tokens
]

print(tokens)
token_ids

['The', 'Camera', 'is', 'Excellent.', 'The', 'Camera', 'is', 'bad.']


[2, 3, 4, 1, 2, 3, 4, 1]

### Till Now what we done 
1. Text -> Vocab -> Tokens -> Token_Ids

- Now if we directly give the Neural Network this Token_ids, that is [2,3,4,5] then the Neural Network may interpret as "excellent"(5) > "is"(4) which is wrong
- So we need Embedding
- For each token_id we further assign the vector , that we gave to the model to Train on
- Lets Create a Embeddinng Layer

In [26]:
import torch
import torch.nn as nn

vocab_size = len(vocab)
embedding_dim = 4 # assume 4 we need to assign embedding for each token_id 

embedding = nn.Embedding(
    vocab_size,
    embedding_dim
)

token_ids = torch.tensor(token_ids)
vectors = embedding(token_ids)

vectors

C:\Users\Navnath\AppData\Local\Temp\ipykernel_5880\2775750610.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  token_ids = torch.tensor(token_ids)


tensor([[-1.4218, -0.9192,  0.2281, -0.1258],
        [-1.0108,  2.1409, -1.6617,  0.9115],
        [-2.0196, -0.6406,  1.2198,  0.5553],
        [ 0.8479,  0.3904, -1.2604, -0.5179],
        [-1.4218, -0.9192,  0.2281, -0.1258],
        [-1.0108,  2.1409, -1.6617,  0.9115],
        [-2.0196, -0.6406,  1.2198,  0.5553],
        [ 0.8479,  0.3904, -1.2604, -0.5179]], grad_fn=<EmbeddingBackward0>)

In [7]:
import torch
import torch.nn as nn

# -------------------------
# 1. Vocabulary
# -------------------------

vocab = {
    "<PAD>": 0,
    "<UNK>": 1,
    "the": 2,
    "camera": 3,
    "is": 4,
    "good": 5,
    "excellent": 6,
    "battery": 7,
    "bad": 8,
    "terrible": 9
}

# -------------------------
# 2. Training data
# -------------------------

texts = [
    [2, 3, 4, 5],  # the camera is good
    [2, 3, 4, 6],  # the camera is excellent
    [2, 7, 4, 8],  # the battery is bad
    [2, 7, 4, 9],  # the battery is terrible
]

labels = [
    1,  # positive
    1,  # positive
    0,  # negative
    0   # negative
]

# -------------------------
# 3. Model
# -------------------------

class SimpleSentimentModel(nn.Module):

    def __init__(self, vocab_size, embedding_dim):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        self.classifier = nn.Linear(
            embedding_dim,
            2
        )

    def forward(self, token_ids):

        vectors = self.embedding(token_ids)

        sentence_vector = vectors.mean(dim=0)

        output = self.classifier(sentence_vector)

        return output


vocab_size = len(vocab)
embedding_dim = 4

model = SimpleSentimentModel(
    vocab_size,
    embedding_dim
)

# -------------------------
# 4. Loss + optimizer
# -------------------------

loss_function = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1
)

# -------------------------
# 5. Save initial weights
# -------------------------

before = model.embedding.weight.detach().clone()

print("Before training:")
print(before)

# -------------------------
# 6. Training
# -------------------------

for epoch in range(100):

    total_loss = 0

    for x, y in zip(texts, labels):

        x = torch.tensor(x, dtype=torch.long)
        y = torch.tensor([y], dtype=torch.long)

        # Forward
        prediction = model(x)

        # Loss
        loss = loss_function(
            prediction.unsqueeze(0),
            y
        )

        # Clear old gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        total_loss += loss.item()

    if epoch % 10 == 0:
        print(
            f"Epoch {epoch}, Loss = {total_loss:.4f}"
        )

# -------------------------
# 7. Compare weights
# -------------------------

after = model.embedding.weight.detach().clone()

print("\nCamera embedding BEFORE:")
print(before[3])

print("\nCamera embedding AFTER:")
print(after[3])

Before training:
tensor([[-0.7971,  0.5296, -0.5590, -1.7275],
        [-0.2290, -0.3730,  0.3297, -0.7725],
        [ 0.9924,  0.4819,  0.3309,  2.5875],
        [ 1.2255,  0.6345,  1.4119,  1.5309],
        [-0.2779,  0.6812, -0.2863,  0.5764],
        [-0.8422,  1.8140, -0.6887,  0.2068],
        [ 1.7072,  0.2554,  0.3984,  0.3602],
        [-0.7360, -0.3272, -0.3220, -0.0947],
        [ 0.3632, -0.7853,  0.4613,  0.3810],
        [ 0.8884, -0.3462, -0.9124, -0.8678]])
Epoch 0, Loss = 2.8939
Epoch 10, Loss = 1.6780
Epoch 20, Loss = 0.8639
Epoch 30, Loss = 0.4647
Epoch 40, Loss = 0.2831
Epoch 50, Loss = 0.1918
Epoch 60, Loss = 0.1403
Epoch 70, Loss = 0.1083
Epoch 80, Loss = 0.0870
Epoch 90, Loss = 0.0721

Camera embedding BEFORE:
tensor([1.2255, 0.6345, 1.4119, 1.5309])

Camera embedding AFTER:
tensor([1.4135, 1.3708, 1.8373, 1.7480])

Camera embedding BEFORE:
tensor([1.2255, 0.6345, 1.4119, 1.5309])

Camera embedding AFTER:
tensor([1.4135, 1.3708, 1.8373, 1.7480])


In [16]:
model_parameters = model.parameters()